# Working with parquet files

## Objective

+ In this assignment, we will use the data downloaded with the module `data_manager` to create features.

(11 pts total)

## Prerequisites

+ This notebook assumes that price data is available to you in the environment variable `PRICE_DATA`. If you have not done so, then execute the notebook `01_materials/labs/2_data_engineering.ipynb` to create this data set.


+ Load the environment variables using dotenv. (1 pt)

In [33]:
from dotenv import load_dotenv

load_dotenv()

import os

print(os.getenv("PRICE_DATA"))


../../05_src/data/prices/


In [43]:
import dask.dataframe as dd

+ Load the environment variable `PRICE_DATA`.
+ Use [glob](https://docs.python.org/3/library/glob.html) to find the path of all parquet files in the directory `PRICE_DATA`.

(1pt)

In [44]:
import os
from glob import glob
PRICE_DATA = os.getenv("PRICE_DATA")
parquet_files = glob(os.path.join(PRICE_DATA, "**/*.parquet"),
                     recursive=True)
print(parquet_files)

print("PRICE_DATA:", PRICE_DATA)
print("number of parquet files:", len(parquet_files))
print("first few files:", parquet_files[:3])

['../../05_src/data/prices\\ACRX\\ACRX_2011\\part.0.parquet', '../../05_src/data/prices\\ACRX\\ACRX_2011\\part.1.parquet', '../../05_src/data/prices\\ACRX\\ACRX_2012\\part.0.parquet', '../../05_src/data/prices\\ACRX\\ACRX_2012\\part.1.parquet', '../../05_src/data/prices\\ACRX\\ACRX_2013\\part.0.parquet', '../../05_src/data/prices\\ACRX\\ACRX_2013\\part.1.parquet', '../../05_src/data/prices\\ACRX\\ACRX_2014\\part.0.parquet', '../../05_src/data/prices\\ACRX\\ACRX_2014\\part.1.parquet', '../../05_src/data/prices\\ACRX\\ACRX_2015\\part.0.parquet', '../../05_src/data/prices\\ACRX\\ACRX_2015\\part.1.parquet', '../../05_src/data/prices\\ACRX\\ACRX_2016\\part.0.parquet', '../../05_src/data/prices\\ACRX\\ACRX_2016\\part.1.parquet', '../../05_src/data/prices\\ACRX\\ACRX_2017\\part.0.parquet', '../../05_src/data/prices\\ACRX\\ACRX_2017\\part.1.parquet', '../../05_src/data/prices\\ACRX\\ACRX_2018\\part.0.parquet', '../../05_src/data/prices\\ACRX\\ACRX_2018\\part.1.parquet', '../../05_src/data/pric

For each ticker and using Dask, do the following:

+ Add lags for variables Close and Adj_Close.
+ Add returns based on Close:
    
    - `returns`: (Close / Close_lag_1) - 1

+ Add the following range: 

    - `hi_lo_range`: this is the day's High minus Low.

+ Assign the result to `dd_feat`.

(4 pt)

In [ ]:
dd_p = dd.read_parquet(parquet_files)
print(dd_p.columns)
dd_p.head(4)

if "Adj_Close" in dd_p.columns:
    adj_col = "Adj_Close"
elif "Adj Close" in dd_p.columns:
    adj_col = "Adj Close"
else:
    raise KeyError("Could not find the adjusted close column in the data. Please check the column names.")
print ("using adjusted close column:", adj_col)

dd_p= dd_p.set_index("ticker")
print(dd_p.head(4))

import pandas as pd
def add_lags(ticker_df):
    ticker_df = ticker_df.sort_values("Date", ascending=True)
    return ticker_df.assign(
           close_lag1=ticker_df["Close"].shift(1),
           adj_close_lag1=ticker_df[adj_col].shift(1))
meta=dd_p._meta.assign(
    close_lag1=pd.Series(dtype="float64"),
    adj_close_lag1=pd.Series(dtype="float64"))

dd_shift = (
    dd_p.groupby("ticker", group_keys=False)
    .apply(add_lags, meta=meta))

dd_feat=dd_shift.assign(
    returns=lambda x: x["Close"] / x["close_lag1"] - 1,
    hi_lo_range=lambda x: x["High"] - x["Low"])
print(dd_feat.head(4))
print(dd_feat.columns)


Index(['Date', 'Open', 'High', 'Low', 'Close', 'Adj Close', 'Volume', 'source',
       'ticker', 'Year'],
      dtype='str')
using adjusted close column: Adj Close
             Date  Open  High   Low  Close  Adj Close     Volume    source  \
ticker                                                                       
ACRX   2011-02-11  5.00  5.09  4.51   4.55       4.55  1899700.0  ACRX.csv   
ACRX   2011-02-14  4.53  4.73  4.35   4.38       4.38    66300.0  ACRX.csv   
ACRX   2011-02-15  4.50  4.50  4.08   4.31       4.31    37400.0  ACRX.csv   
ACRX   2011-02-16  4.37  4.37  4.15   4.24       4.24     9500.0  ACRX.csv   

        Year  
ticker        
ACRX    2011  
ACRX    2011  
ACRX    2011  
ACRX    2011  
Index(['Date', 'Open', 'High', 'Low', 'Close', 'Adj Close', 'Volume', 'source',
       'Year', 'close_lag1', 'adj_close_lag1', 'returns', 'hi_lo_range'],
      dtype='str')
             Date    Open    High     Low   Close  Adj Close   Volume  \
ticker                         

+ Convert the Dask data frame to a pandas data frame. 
+ Add a new feature containing the moving average of `returns` using a window of 10 days. There are several ways to solve this task, a simple one uses `.rolling(10).mean()`.

(3 pt)

In [42]:
# Write your code below.
df_feat = dd_feat.compute().reset_index()
df_feat = df_feat.sort_values(["ticker", "Date"])
df_feat["returns_ma_10"] = (
df_feat.groupby("ticker")["returns"]
    .transform(lambda s: s.rolling(10).mean()))

df_feat[
    ["ticker", "Date", "returns", "returns_ma_10"]].head(20)


,ticker,Date,returns,returns_ma_10
0,ACRX,2011-02-11,NaN,NaN
1,ACRX,2011-02-14,-0.037363,NaN
2,ACRX,2011-02-15,-0.015982,NaN
3,ACRX,2011-02-16,-0.016241,NaN
4,ACRX,2011-02-17,-0.091981,NaN
5,ACRX,2011-02-18,-0.051948,NaN
6,ACRX,2011-02-22,0.057534,NaN
7,ACRX,2011-02-23,0.025907,NaN
8,ACRX,2011-02-24,-0.053030,NaN
9,ACRX,2011-02-25,-0.042667,NaN


Please comment:

+ Was it necessary to convert to pandas to calculate the moving average return?
+ Would it have been better to do it in Dask? Why?

(1 pt)
No it was not required. Dask calculated the rolling averages. 
pandas work better when the data is small and Dask is more suitable when datasets are large because it can process data in parts. 

## Criteria

The [rubric](./assignment_1_rubric_clean.xlsx) contains the criteria for grading.

## Submission Information

🚨 **Please review our [Assignment Submission Guide](https://github.com/UofT-DSI/onboarding/blob/main/onboarding_documents/submissions.md)** 🚨 for detailed instructions on how to format, branch, and submit your work. Following these guidelines is crucial for your submissions to be evaluated correctly.

### Submission Parameters:
* Submission Due Date: `HH:MM AM/PM - DD/MM/YYYY`
* The branch name for your repo should be: `assignment-1`
* What to submit for this assignment:
    * This Jupyter Notebook (assignment_1.ipynb) should be populated and should be the only change in your pull request.
* What the pull request link should look like for this assignment: `https://github.com/<your_github_username>/production/pull/<pr_id>`
    * Open a private window in your browser. Copy and paste the link to your pull request into the address bar. Make sure you can see your pull request properly. This helps the technical facilitator and learning support staff review your submission easily.

Checklist:
- [ ] Created a branch with the correct naming convention.
- [ ] Ensured that the repository is public.
- [ ] Reviewed the PR description guidelines and adhered to them.
- [ ] Verify that the link is accessible in a private browser window.

If you encounter any difficulties or have questions, please don't hesitate to reach out to our team via our Slack at `#cohort-3-help`. Our Technical Facilitators and Learning Support staff are here to help you navigate any challenges.